In [7]:
# Install Required Dependencies
import subprocess
import sys

def install_package(package):
    """Install a package using pip"""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ Successfully installed {package}")
    except subprocess.CalledProcessError as e:
        print(f"❌ Failed to install {package}: {e}")

# List of required packages
required_packages = [
    "numpy",
    "pandas", 
    "matplotlib",
    "seaborn",
    "Pillow",  # PIL
    "ipywidgets",
    "tqdm",
    "requests",
    "scipy",  # for .mat file loading
    "torch",  # PyTorch
    "torchvision",  # torchvision datasets
    "kaggle",  # for Kaggle downloads (optional)
    "kagglehub",  # for Kaggle dataset downloads
    "datasets"
]

print("🔧 Installing required packages for ImageNet Dataset Explorer...")
print("="*60)

for package in required_packages:
    install_package(package)

print("\n🎉 Installation complete!")
print("📋 If you see any errors above, you may need to install those packages manually")
print("💡 Restart the kernel after installation to ensure all packages are available")

🔧 Installing required packages for ImageNet Dataset Explorer...
✅ Successfully installed numpy
✅ Successfully installed numpy
✅ Successfully installed pandas
✅ Successfully installed pandas
✅ Successfully installed matplotlib
✅ Successfully installed matplotlib
✅ Successfully installed seaborn
✅ Successfully installed seaborn
✅ Successfully installed Pillow
✅ Successfully installed Pillow
✅ Successfully installed ipywidgets
✅ Successfully installed ipywidgets
✅ Successfully installed tqdm
✅ Successfully installed tqdm
✅ Successfully installed requests
✅ Successfully installed requests
✅ Successfully installed scipy
✅ Successfully installed scipy
✅ Successfully installed torch
✅ Successfully installed torch
✅ Successfully installed torchvision
✅ Successfully installed torchvision
✅ Successfully installed kaggle
✅ Successfully installed kaggle
✅ Successfully installed kagglehub
✅ Successfully installed kagglehub
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 28.3 MB/s  0:00:0

# ImageNet-1K Dataset Explorer 📊

This notebook provides comprehensive exploration and visualization tools for the ImageNet-1K dataset. You can:

- 🔍 Browse the dataset structure and class distribution
- 🖼️ View sample images from different classes
- 📈 Analyze image properties and metadata
- 🎛️ Use interactive widgets to explore the dataset
- 📊 Generate detailed statistics and visualizations

## Requirements
- ImageNet-1K dataset downloaded and organized in standard format
- Python libraries: PIL, matplotlib, pandas, numpy, ipywidgets
- Jupyter notebook environment

In [8]:
# Import Required Libraries
import os
import sys
import random
import json
from pathlib import Path
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')

# Image processing and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ExifTags
import seaborn as sns

# Interactive widgets
try:
    import ipywidgets as widgets
    from IPython.display import display, HTML
    WIDGETS_AVAILABLE = True
    print("✅ Interactive widgets available")
except ImportError:
    WIDGETS_AVAILABLE = False
    print("⚠️ ipywidgets not available. Install with: pip install ipywidgets")

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

print("📚 All required libraries imported successfully!")

✅ Interactive widgets available
📚 All required libraries imported successfully!


In [4]:
# Setup Dataset Path and Configuration
# ⚠️ UPDATE THIS PATH TO YOUR IMAGENET DATASET LOCATION
IMAGENET_ROOT = r"./datasets/imagenet1k"  # Default path for downloaded dataset

# Alternative paths you might need to try:
# IMAGENET_ROOT = r"D:\datasets\imagenet"
# IMAGENET_ROOT = r"/data/imagenet"
# IMAGENET_ROOT = r"/home/user/datasets/imagenet"
# IMAGENET_ROOT = r".\datasets\imagenet1k"  # Windows path

TRAIN_DIR = os.path.join(IMAGENET_ROOT, "train")
VAL_DIR = os.path.join(IMAGENET_ROOT, "val")

# Configuration settings
MAX_IMAGES_PER_CLASS = 10  # Max images to display per class
FIGURE_SIZE = (15, 10)     # Default figure size
THUMBNAIL_SIZE = (224, 224) # Thumbnail size for previews

print(f"📁 Dataset root: {IMAGENET_ROOT}")
print(f"📁 Train directory: {TRAIN_DIR}")
print(f"📁 Validation directory: {VAL_DIR}")

# Check if directories exist
if os.path.exists(TRAIN_DIR):
    print("✅ Training directory found")
    # Count classes and images
    train_classes = [d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))]
    print(f"📊 Training classes: {len(train_classes)}")
else:
    print("❌ Training directory not found")
    print("💡 Use the download cells above to get the ImageNet dataset")

if os.path.exists(VAL_DIR):
    print("✅ Validation directory found")
    # Count classes and images
    val_classes = [d for d in os.listdir(VAL_DIR) if os.path.isdir(os.path.join(VAL_DIR, d))]
    print(f"📊 Validation classes: {len(val_classes)}")
else:
    print("❌ Validation directory not found")
    print("💡 Use the download cells above to get the ImageNet dataset")

# Quick dataset info
if os.path.exists(IMAGENET_ROOT):
    total_size = sum(f.stat().st_size for f in Path(IMAGENET_ROOT).rglob('*') if f.is_file())
    print(f"💾 Total dataset size: {total_size / (1024**3):.1f} GB")
else:
    print("📥 Dataset not found - ready to download!")

📁 Dataset root: ./datasets/imagenet1k
📁 Train directory: ./datasets/imagenet1k/train
📁 Validation directory: ./datasets/imagenet1k/val
❌ Training directory not found
💡 Use the download cells above to get the ImageNet dataset
❌ Validation directory not found
💡 Use the download cells above to get the ImageNet dataset
📥 Dataset not found - ready to download!


In [2]:
!kaggle competitions download -c imagenet-object-localization-challenge

User cancelled operation
^C


In [ ]:
# Monitor Kaggle Download Progress
import os
import time
import subprocess
from pathlib import Path

def check_download_status():
    """Check the current status of ImageNet download"""
    
    # Check current directory for downloaded files
    current_dir = Path(".")
    
    print("🔍 Checking download status...")
    print("="*50)
    
    # Look for ImageNet related files
    imagenet_files = []
    for file_pattern in ["*imagenet*", "*ILSVRC*", "*.zip", "*.tar", "*.tar.gz"]:
        imagenet_files.extend(list(current_dir.glob(file_pattern)))
    
    if imagenet_files:
        print("📁 Found ImageNet related files:")
        total_size = 0
        for file_path in imagenet_files:
            if file_path.is_file():
                size_gb = file_path.stat().st_size / (1024**3)
                total_size += size_gb
                print(f"   📄 {file_path.name}: {size_gb:.2f} GB")
        
        print(f"\n📊 Total downloaded: {total_size:.2f} GB")
        
        # Expected ImageNet size is around 150GB compressed
        expected_size = 150
        progress = (total_size / expected_size) * 100
        print(f"📈 Estimated progress: {progress:.1f}%")
        
        if progress >= 100:
            print("✅ Download appears complete!")
        else:
            print(f"⏳ Download in progress... ({100-progress:.1f}% remaining)")
    else:
        print("❌ No ImageNet files found in current directory")
        print("💡 Download may not have started yet or files are in a different location")
    
    # Check if kaggle process is running
    try:
        result = subprocess.run(["pgrep", "-f", "kaggle"], capture_output=True, text=True)
        if result.stdout.strip():
            print("\n🔄 Kaggle download process is running")
            print("   Process IDs:", result.stdout.strip())
        else:
            print("\n⏸️ No active kaggle download process found")
    except Exception as e:
        print(f"\n⚠️ Could not check process status: {e}")
    
    return imagenet_files

def monitor_download_progress(check_interval=30):
    """Monitor download progress with periodic updates"""
    
    print("📊 Starting download progress monitor...")
    print(f"🔄 Checking every {check_interval} seconds (Press Ctrl+C to stop)")
    print("="*60)
    
    try:
        while True:
            files = check_download_status()
            
            if files:
                # Check if download is still active
                result = subprocess.run(["pgrep", "-f", "kaggle"], capture_output=True, text=True)
                if not result.stdout.strip():
                    print("✅ Download process completed!")
                    break
            
            print(f"\n⏰ Next check in {check_interval} seconds...")
            time.sleep(check_interval)
            
    except KeyboardInterrupt:
        print("\n🛑 Monitoring stopped by user")
    except Exception as e:
        print(f"\n❌ Error during monitoring: {e}")

# Run initial status check
check_download_status()

In [ ]:
# Alternative Methods to Check Download Progress

def check_network_activity():
    """Check if there's active network download"""
    try:
        # Check network connections
        result = subprocess.run(["netstat", "-i"], capture_output=True, text=True)
        print("🌐 Network Interface Activity:")
        print(result.stdout)
    except:
        print("⚠️ Could not check network activity")

def watch_file_growth():
    """Watch file size changes in real-time"""
    current_dir = Path(".")
    
    print("👀 Watching for file size changes...")
    print("Press Ctrl+C to stop")
    
    # Get initial file sizes
    initial_sizes = {}
    for file_path in current_dir.glob("*"):
        if file_path.is_file():
            initial_sizes[file_path.name] = file_path.stat().st_size
    
    try:
        while True:
            time.sleep(5)
            current_sizes = {}
            
            for file_path in current_dir.glob("*"):
                if file_path.is_file():
                    current_size = file_path.stat().st_size
                    current_sizes[file_path.name] = current_size
                    
                    # Check if file size changed
                    if file_path.name in initial_sizes:
                        size_diff = current_size - initial_sizes[file_path.name]
                        if size_diff > 0:
                            print(f"📈 {file_path.name}: +{size_diff / (1024**2):.2f} MB")
            
            initial_sizes = current_sizes.copy()
            
    except KeyboardInterrupt:
        print("\n🛑 File watching stopped")

# Quick commands to check download manually
print("🔧 Manual Commands to Check Download Status:")
print("="*50)
print("1. Check files in current directory:")
print("   !ls -lh *imagenet* *ILSVRC* *.zip *.tar*")
print("")
print("2. Check disk usage:")
print("   !du -sh .")
print("")
print("3. Check running processes:")
print("   !ps aux | grep kaggle")
print("")
print("4. Watch file sizes in real-time:")
print("   !watch -n 5 'ls -lh *.zip *.tar*'")
print("")
print("5. Monitor network activity:")
print("   !iostat -x 1")
print("")
print("💡 Run these commands in separate cells or terminal")

## 📥 ImageNet-1K Dataset Download & Setup

⚠️ **Important Notes:**
- ImageNet-1K is ~150GB compressed, ~500GB extracted
- Requires registration at [image-net.org](https://image-net.org/download.php)
- Download can take hours depending on connection speed
- Consider using cloud storage or institutional access if available

In [5]:
# ImageNet-1K Dataset Downloader and Extractor
import subprocess
import tarfile
import zipfile
import urllib.request
from tqdm.auto import tqdm
import requests
import hashlib

class ImageNetDownloader:
    """Download and extract ImageNet-1K dataset"""
    
    def __init__(self, download_dir="./datasets/imagenet1k"):
        self.download_dir = Path(download_dir)
        self.download_dir.mkdir(parents=True, exist_ok=True)
        
        # Official ImageNet URLs (requires authentication)
        self.urls = {
            'train': 'https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_train.tar',
            'val': 'https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar',
            'devkit': 'https://image-net.org/data/ILSVRC/2012/ILSVRC2012_devkit_t12.tar.gz'
        }
        
        # File sizes for verification
        self.file_sizes = {
            'train': 147897477120,  # ~138GB
            'val': 6744924160,      # ~6.3GB
            'devkit': 2567563       # ~2.4MB
        }
        
        # Expected checksums (MD5)
        self.checksums = {
            'train': '1d675b47d978889d74fa0da5fadfb00e',
            'val': '29b22e2961454d5413ddabcf34fc5622',
            'devkit': 'fa75699e90414af021442c21a62c3abf'
        }
    
    def download_with_progress(self, url, filename, expected_size=None):
        """Download file with progress bar"""
        filepath = self.download_dir / filename
        
        if filepath.exists():
            print(f"✅ {filename} already exists")
            return filepath
        
        print(f"📥 Downloading {filename}...")
        
        try:
            response = requests.get(url, stream=True)
            response.raise_for_status()
            
            total_size = int(response.headers.get('content-length', 0))
            if expected_size and total_size != expected_size:
                print(f"⚠️ Warning: Expected size {expected_size}, got {total_size}")
            
            with open(filepath, 'wb') as f:
                with tqdm(total=total_size, unit='B', unit_scale=True, desc=filename) as pbar:
                    for chunk in response.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))
            
            print(f"✅ Downloaded {filename}")
            return filepath
            
        except Exception as e:
            print(f"❌ Error downloading {filename}: {e}")
            if filepath.exists():
                filepath.unlink()
            return None
    
    def verify_checksum(self, filepath, expected_md5):
        """Verify file integrity using MD5 checksum"""
        print(f"🔍 Verifying {filepath.name}...")
        
        hash_md5 = hashlib.md5()
        with open(filepath, "rb") as f:
            for chunk in iter(lambda: f.read(4096), b""):
                hash_md5.update(chunk)
        
        actual_md5 = hash_md5.hexdigest()
        if actual_md5 == expected_md5:
            print(f"✅ Checksum verified for {filepath.name}")
            return True
        else:
            print(f"❌ Checksum mismatch for {filepath.name}")
            print(f"   Expected: {expected_md5}")
            print(f"   Actual:   {actual_md5}")
            return False
    
    def extract_tar(self, tar_path, extract_to):
        """Extract tar file with progress"""
        print(f"📦 Extracting {tar_path.name}...")
        
        try:
            with tarfile.open(tar_path, 'r') as tar:
                members = tar.getmembers()
                
                with tqdm(total=len(members), desc="Extracting") as pbar:
                    for member in members:
                        tar.extract(member, extract_to)
                        pbar.update(1)
            
            print(f"✅ Extracted {tar_path.name}")
            return True
            
        except Exception as e:
            print(f"❌ Error extracting {tar_path.name}: {e}")
            return False
    
    def organize_training_data(self):
        """Organize training data into class folders"""
        train_dir = self.download_dir / "train"
        
        if not train_dir.exists():
            print("❌ Training directory not found")
            return False
        
        print("🗂️ Organizing training data into class folders...")
        
        # Get all tar files in train directory
        tar_files = list(train_dir.glob("*.tar"))
        
        if not tar_files:
            print("✅ Training data already organized")
            return True
        
        for tar_file in tqdm(tar_files, desc="Organizing classes"):
            class_name = tar_file.stem  # e.g., "n01440764"
            class_dir = train_dir / class_name
            class_dir.mkdir(exist_ok=True)
            
            try:
                with tarfile.open(tar_file, 'r') as tar:
                    tar.extractall(class_dir)
                
                # Remove the tar file after successful extraction
                tar_file.unlink()
                
            except Exception as e:
                print(f"❌ Error extracting {tar_file}: {e}")
                continue
        
        print("✅ Training data organized")
        return True
    
    def setup_validation_data(self):
        """Setup validation data with proper class folders"""
        val_dir = self.download_dir / "val"
        devkit_dir = self.download_dir / "ILSVRC2012_devkit_t12"
        
        if not val_dir.exists() or not devkit_dir.exists():
            print("❌ Validation data or devkit not found")
            return False
        
        # Check if already organized
        val_subdirs = [d for d in val_dir.iterdir() if d.is_dir()]
        if len(val_subdirs) > 10:  # Assume organized if many subdirectories
            print("✅ Validation data already organized")
            return True
        
        print("🗂️ Organizing validation data...")
        
        # Load validation labels
        labels_file = devkit_dir / "data" / "ILSVRC2012_validation_ground_truth.txt"
        synsets_file = devkit_dir / "data" / "meta.mat"
        
        if not labels_file.exists():
            print("❌ Validation labels file not found")
            return False
        
        try:
            # Read validation labels
            with open(labels_file, 'r') as f:
                val_labels = [int(line.strip()) for line in f]
            
            # Get class names (requires scipy for .mat files)
            try:
                from scipy.io import loadmat
                meta = loadmat(synsets_file)
                synsets = meta['synsets']
                class_names = [str(synsets[i][0][1][0]) for i in range(1000)]
            except ImportError:
                print("⚠️ scipy not available, using generic class names")
                class_names = [f"class_{i:04d}" for i in range(1000)]
            
            # Create class directories
            for class_name in set(class_names):
                (val_dir / class_name).mkdir(exist_ok=True)
            
            # Move validation images to class folders
            val_images = sorted(val_dir.glob("ILSVRC2012_val_*.JPEG"))
            
            for i, img_path in enumerate(tqdm(val_images, desc="Moving validation images")):
                if i < len(val_labels):
                    class_idx = val_labels[i] - 1  # Convert to 0-indexed
                    class_name = class_names[class_idx]
                    new_path = val_dir / class_name / img_path.name
                    img_path.rename(new_path)
            
            print("✅ Validation data organized")
            return True
            
        except Exception as e:
            print(f"❌ Error organizing validation data: {e}")
            return False
    
    def download_and_setup(self, download_train=True, download_val=True, verify_checksums=True):
        """Complete download and setup process"""
        print("🚀 Starting ImageNet-1K download and setup...")
        print(f"📁 Download directory: {self.download_dir}")
        
        success = True
        
        # Download devkit first (needed for validation setup)
        print("\n1️⃣ Downloading development kit...")
        devkit_file = self.download_with_progress(
            self.urls['devkit'], 
            'ILSVRC2012_devkit_t12.tar.gz',
            self.file_sizes['devkit']
        )
        
        if devkit_file and verify_checksums:
            if not self.verify_checksum(devkit_file, self.checksums['devkit']):
                success = False
        
        if devkit_file:
            self.extract_tar(devkit_file, self.download_dir)
        
        # Download and setup training data
        if download_train:
            print("\n2️⃣ Downloading training data...")
            train_file = self.download_with_progress(
                self.urls['train'], 
                'ILSVRC2012_img_train.tar',
                self.file_sizes['train']
            )
            
            if train_file and verify_checksums:
                if not self.verify_checksum(train_file, self.checksums['train']):
                    success = False
            
            if train_file:
                train_extract_dir = self.download_dir / "train"
                train_extract_dir.mkdir(exist_ok=True)
                
                if self.extract_tar(train_file, train_extract_dir):
                    self.organize_training_data()
        
        # Download and setup validation data
        if download_val:
            print("\n3️⃣ Downloading validation data...")
            val_file = self.download_with_progress(
                self.urls['val'], 
                'ILSVRC2012_img_val.tar',
                self.file_sizes['val']
            )
            
            if val_file and verify_checksums:
                if not self.verify_checksum(val_file, self.checksums['val']):
                    success = False
            
            if val_file:
                val_extract_dir = self.download_dir / "val"
                val_extract_dir.mkdir(exist_ok=True)
                
                if self.extract_tar(val_file, val_extract_dir):
                    self.setup_validation_data()
        
        if success:
            print("\n🎉 ImageNet-1K download and setup completed successfully!")
        else:
            print("\n⚠️ Download completed with some issues. Please check the output above.")
        
        return success
    
    def get_dataset_info(self):
        """Get information about the downloaded dataset"""
        train_dir = self.download_dir / "train"
        val_dir = self.download_dir / "val"
        
        info = {
            'train_classes': 0,
            'train_images': 0,
            'val_classes': 0,
            'val_images': 0,
            'total_size_gb': 0
        }
        
        # Count training data
        if train_dir.exists():
            train_classes = [d for d in train_dir.iterdir() if d.is_dir()]
            info['train_classes'] = len(train_classes)
            
            for class_dir in train_classes:
                images = list(class_dir.glob("*.JPEG"))
                info['train_images'] += len(images)
        
        # Count validation data
        if val_dir.exists():
            val_classes = [d for d in val_dir.iterdir() if d.is_dir()]
            info['val_classes'] = len(val_classes)
            
            for class_dir in val_classes:
                images = list(class_dir.glob("*.JPEG"))
                info['val_images'] += len(images)
        
        # Calculate total size
        if self.download_dir.exists():
            total_bytes = sum(f.stat().st_size for f in self.download_dir.rglob('*') if f.is_file())
            info['total_size_gb'] = total_bytes / (1024**3)
        
        return info

# Create downloader instance
downloader = ImageNetDownloader()

print("📋 ImageNet-1K Downloader Ready!")
print("⚠️ Note: You need to register at image-net.org to get download access")
print("🔗 Registration: https://image-net.org/download.php")

📋 ImageNet-1K Downloader Ready!
⚠️ Note: You need to register at image-net.org to get download access
🔗 Registration: https://image-net.org/download.php


In [ ]:
# Alternative: Download using PyTorch/Torchvision (Easier but slower)
def download_imagenet_torchvision(root_dir="./datasets", download=True):
    """
    Download ImageNet using torchvision (requires authentication)
    This is easier but slower than manual download
    """
    try:
        import torchvision.datasets as datasets
        import torchvision.transforms as transforms
        
        print("📥 Downloading ImageNet using torchvision...")
        
        # Note: This requires you to manually download and place the files
        # torchvision doesn't automatically download ImageNet due to licensing
        
        transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
        
        # Training dataset
        train_dataset = datasets.ImageNet(
            root=root_dir,
            split='train',
            transform=transform,
            download=False  # Set to True if files are manually placed
        )
        
        # Validation dataset  
        val_dataset = datasets.ImageNet(
            root=root_dir,
            split='val',
            transform=transform,
            download=False  # Set to True if files are manually placed
        )
        
        print(f"✅ ImageNet loaded successfully!")
        print(f"📊 Training samples: {len(train_dataset)}")
        print(f"📊 Validation samples: {len(val_dataset)}")
        
        return train_dataset, val_dataset
        
    except Exception as e:
        print(f"❌ Error loading ImageNet with torchvision: {e}")
        print("💡 You may need to manually download the dataset files first")
        return None, None

# Alternative download methods
def download_imagenet_kaggle():
    """Download ImageNet from Kaggle (if available)"""
    print("📥 Downloading from Kaggle...")
    print("🔑 Make sure you have Kaggle API configured:")
    print("   1. Install: pip install kaggle")
    print("   2. Get API key from https://www.kaggle.com/account")
    print("   3. Place kaggle.json in ~/.kaggle/")
    
    try:
        import kaggle
        
        # Download ImageNet dataset from Kaggle
        kaggle.api.dataset_download_files(
            'imagenet-object-localization-challenge',
            path='./datasets/',
            unzip=True
        )
        
        print("✅ Downloaded from Kaggle")
        
    except Exception as e:
        print(f"❌ Kaggle download failed: {e}")

def download_imagenet_aws():
    """Download ImageNet from AWS Open Data"""
    print("📥 Downloading from AWS Open Data...")
    print("🔗 AWS ImageNet: https://aws.amazon.com/opendata/imagenet/")
    
    aws_commands = [
        "# Install AWS CLI",
        "pip install awscli",
        "",
        "# Download training data",
        "aws s3 sync s3://imagenet-dataset/train ./datasets/imagenet/train --no-sign-request",
        "",
        "# Download validation data", 
        "aws s3 sync s3://imagenet-dataset/val ./datasets/imagenet/val --no-sign-request"
    ]
    
    print("💻 Run these commands in terminal:")
    for cmd in aws_commands:
        print(f"   {cmd}")

# Display download options
print("🎯 ImageNet-1K Download Options:")
print("="*50)
print("1️⃣ Manual Download (Recommended)")
print("   - Register at image-net.org")
print("   - Use ImageNetDownloader class above")
print("   - Most reliable method")
print("")
print("2️⃣ Torchvision (Requires manual file placement)")
print("   - download_imagenet_torchvision()")
print("   - Easier setup but requires manual download")
print("")
print("3️⃣ Kaggle (If available)")
print("   - download_imagenet_kaggle()")
print("   - Requires Kaggle account and API setup")
print("")
print("4️⃣ AWS Open Data")
print("   - download_imagenet_aws()")
print("   - Free but requires AWS CLI")
print("")
print("💡 Recommendation: Use option 1 (Manual Download) for best results")

In [ ]:
# Quick Start: Download ImageNet-1K
def quick_download_setup():
    """Interactive setup for ImageNet download"""
    
    print("🚀 ImageNet-1K Quick Setup")
    print("="*40)
    
    # Check available space
    import shutil
    free_space_gb = shutil.disk_usage('.')[2] / (1024**3)
    print(f"💾 Available disk space: {free_space_gb:.1f} GB")
    
    if free_space_gb < 600:
        print("⚠️ Warning: ImageNet requires ~500GB+ space")
        print("💡 Consider using cloud storage or external drive")
    
    # Setup download directory
    download_dir = input("📁 Enter download directory [./datasets/imagenet1k]: ").strip()
    if not download_dir:
        download_dir = "./datasets/imagenet1k"
    
    print(f"📂 Using directory: {download_dir}")
    
    # Create downloader
    downloader = ImageNetDownloader(download_dir)
    
    # Ask what to download
    print("\n📋 What would you like to download?")
    print("1. Training data only (~138GB)")
    print("2. Validation data only (~6GB)")  
    print("3. Both training and validation (~144GB)")
    print("4. Check existing dataset")
    
    choice = input("Enter choice [1-4]: ").strip()
    
    if choice == "1":
        print("📥 Starting training data download...")
        downloader.download_and_setup(download_train=True, download_val=False)
    elif choice == "2":
        print("📥 Starting validation data download...")
        downloader.download_and_setup(download_train=False, download_val=True)
    elif choice == "3":
        print("📥 Starting full dataset download...")
        downloader.download_and_setup(download_train=True, download_val=True)
    elif choice == "4":
        print("📊 Checking existing dataset...")
        info = downloader.get_dataset_info()
        print(f"Training: {info['train_classes']} classes, {info['train_images']} images")
        print(f"Validation: {info['val_classes']} classes, {info['val_images']} images")
        print(f"Total size: {info['total_size_gb']:.1f} GB")
    else:
        print("❌ Invalid choice")
        return
    
    print("\n✅ Setup complete!")
    return downloader

# Run quick setup if desired
print("💡 Run quick_download_setup() to start interactive download")
print("📋 Or use the ImageNetDownloader class directly for custom setup")

# Example usage
print("\n📖 Example Usage:")
print("```python")
print("# Quick interactive setup")
print("downloader = quick_download_setup()")
print("")
print("# Or manual setup")
print("downloader = ImageNetDownloader('./my_imagenet')")
print("downloader.download_and_setup()")
print("")
print("# Check dataset info")
print("info = downloader.get_dataset_info()")
print("print(info)")
print("```")

In [ ]:
# Create PyTorch DataLoader for Training
def create_imagenet_dataloaders(data_dir, batch_size=32, num_workers=4, img_size=224):
    """Create PyTorch DataLoaders for ImageNet training"""
    
    import torch
    from torch.utils.data import DataLoader
    import torchvision.transforms as transforms
    import torchvision.datasets as datasets
    
    # Training transforms with data augmentation
    train_transforms = transforms.Compose([
        transforms.RandomResizedCrop(img_size),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    # Validation transforms (no augmentation)
    val_transforms = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    # Create datasets
    train_dataset = datasets.ImageFolder(
        root=os.path.join(data_dir, 'train'),
        transform=train_transforms
    )
    
    val_dataset = datasets.ImageFolder(
        root=os.path.join(data_dir, 'val'),
        transform=val_transforms
    )
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )
    
    print(f"📊 Created DataLoaders:")
    print(f"   Training: {len(train_dataset)} samples, {len(train_loader)} batches")
    print(f"   Validation: {len(val_dataset)} samples, {len(val_loader)} batches")
    print(f"   Classes: {len(train_dataset.classes)}")
    print(f"   Batch size: {batch_size}")
    
    return train_loader, val_loader, train_dataset.classes

# Example model training setup
def setup_imagenet_training():
    """Complete setup for ImageNet training"""
    
    print("🏋️ ImageNet Training Setup")
    print("="*30)
    
    # Check if dataset exists
    data_dir = "./datasets/imagenet1k"
    train_dir = os.path.join(data_dir, "train")
    val_dir = os.path.join(data_dir, "val")
    
    if not os.path.exists(train_dir) or not os.path.exists(val_dir):
        print("❌ ImageNet dataset not found!")
        print("📥 Please download the dataset first using the downloader above")
        return None, None, None
    
    print("✅ Dataset found!")
    
    # Create data loaders
    try:
        train_loader, val_loader, classes = create_imagenet_dataloaders(
            data_dir=data_dir,
            batch_size=32,  # Adjust based on GPU memory
            num_workers=4,
            img_size=224
        )
        
        print("\n🎯 Ready for training!")
        print("💡 Use these loaders with your PyTorch model")
        
        return train_loader, val_loader, classes
        
    except Exception as e:
        print(f"❌ Error creating data loaders: {e}")
        return None, None, None

# Memory optimization tips
print("💾 Memory Optimization Tips for ImageNet:")
print("="*45)
print("1️⃣ Use smaller batch sizes (16-32) if GPU memory is limited")
print("2️⃣ Enable mixed precision training (torch.cuda.amp)")
print("3️⃣ Use gradient checkpointing for large models")
print("4️⃣ Consider using smaller image sizes (192 instead of 224)")
print("5️⃣ Use multiple GPUs with DataParallel or DistributedDataParallel")
print("6️⃣ Enable pin_memory=True for faster data transfer")
print("7️⃣ Use SSD storage for faster data loading")

# Show example training loop
print("\n📝 Example Training Code:")
print("```python")
print("# Setup")
print("train_loader, val_loader, classes = setup_imagenet_training()")
print("")
print("# Create model (example with ResNet)")
print("import torchvision.models as models")
print("model = models.resnet50(pretrained=False, num_classes=1000)")
print("model = model.cuda()")
print("")
print("# Training loop")
print("for epoch in range(num_epochs):")
print("    for batch_idx, (data, target) in enumerate(train_loader):")
print("        data, target = data.cuda(), target.cuda()")
print("        # ... your training code ...")
print("```")

In [ ]:
# Load ImageNet Class Labels
def load_imagenet_classes():
    """Load ImageNet class labels and create mappings"""
    
    # ImageNet 1000 class labels (sample - you can download the full file)
    # Full file available at: https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt
    
    # Create a basic mapping for demonstration
    # In practice, you'd load this from a file or use torchvision.datasets.ImageNet
    
    classes = {}
    idx_to_class = {}
    
    try:
        # Try to get classes from the dataset directory structure
        if os.path.exists(TRAIN_DIR):
            class_dirs = sorted([d for d in os.listdir(TRAIN_DIR) 
                               if os.path.isdir(os.path.join(TRAIN_DIR, d))])
            
            for idx, class_dir in enumerate(class_dirs):
                classes[class_dir] = idx
                idx_to_class[idx] = class_dir
            
            print(f"✅ Loaded {len(classes)} classes from dataset structure")
            
        else:
            print("⚠️ No dataset directory found, using sample classes")
            # Sample classes for demonstration
            sample_classes = [
                "n01440764", "n01443537", "n01484850", "n01491361", "n01494475",
                "n01496331", "n01498041", "n01514668", "n01514859", "n01518878"
            ]
            for idx, class_id in enumerate(sample_classes):
                classes[class_id] = idx
                idx_to_class[idx] = class_id
                
    except Exception as e:
        print(f"❌ Error loading classes: {e}")
        return {}, {}
    
    return classes, idx_to_class

# Load the classes
class_to_idx, idx_to_class = load_imagenet_classes()
num_classes = len(class_to_idx)

print(f"📊 Total classes: {num_classes}")
if num_classes > 0:
    print(f"📝 Sample classes: {list(class_to_idx.keys())[:5]}...")
else:
    print("❌ No classes loaded. Please check your dataset path.")

In [ ]:
# Browse Dataset Structure
def analyze_dataset_structure():
    """Analyze the structure of the ImageNet dataset"""
    
    structure_info = {
        'train': {},
        'val': {},
        'total_train_images': 0,
        'total_val_images': 0
    }
    
    print("🔍 Analyzing dataset structure...")
    
    # Analyze training data
    if os.path.exists(TRAIN_DIR):
        print("📁 Scanning training directory...")
        train_classes = os.listdir(TRAIN_DIR)
        
        for class_dir in train_classes[:20]:  # Analyze first 20 classes for speed
            class_path = os.path.join(TRAIN_DIR, class_dir)
            if os.path.isdir(class_path):
                images = [f for f in os.listdir(class_path) 
                         if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
                structure_info['train'][class_dir] = len(images)
                structure_info['total_train_images'] += len(images)
        
        print(f"✅ Analyzed {len(structure_info['train'])} training classes")
    
    # Analyze validation data
    if os.path.exists(VAL_DIR):
        print("📁 Scanning validation directory...")
        val_classes = os.listdir(VAL_DIR)
        
        for class_dir in val_classes[:20]:  # Analyze first 20 classes for speed
            class_path = os.path.join(VAL_DIR, class_dir)
            if os.path.isdir(class_path):
                images = [f for f in os.listdir(class_path) 
                         if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
                structure_info['val'][class_dir] = len(images)
                structure_info['total_val_images'] += len(images)
        
        print(f"✅ Analyzed {len(structure_info['val'])} validation classes")
    
    return structure_info

# Run the analysis
dataset_info = analyze_dataset_structure()

# Display results
print("\n📊 Dataset Structure Summary:")
print(f"🏋️ Training classes analyzed: {len(dataset_info['train'])}")
print(f"🏋️ Total training images (sample): {dataset_info['total_train_images']}")
print(f"✅ Validation classes analyzed: {len(dataset_info['val'])}")
print(f"✅ Total validation images (sample): {dataset_info['total_val_images']}")

if dataset_info['train']:
    train_counts = list(dataset_info['train'].values())
    print(f"📈 Training images per class: min={min(train_counts)}, max={max(train_counts)}, avg={np.mean(train_counts):.1f}")

if dataset_info['val']:
    val_counts = list(dataset_info['val'].values())
    print(f"📈 Validation images per class: min={min(val_counts)}, max={max(val_counts)}, avg={np.mean(val_counts):.1f}")

In [ ]:
# Display Sample Images by Class
def display_sample_images(class_name, num_images=6, split='train'):
    """Display sample images from a specific class"""
    
    data_dir = TRAIN_DIR if split == 'train' else VAL_DIR
    class_path = os.path.join(data_dir, class_name)
    
    if not os.path.exists(class_path):
        print(f"❌ Class '{class_name}' not found in {split} directory")
        return
    
    # Get image files
    image_files = [f for f in os.listdir(class_path) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    
    if len(image_files) == 0:
        print(f"❌ No images found in class '{class_name}'")
        return
    
    # Randomly sample images
    sample_images = random.sample(image_files, min(num_images, len(image_files)))
    
    # Create subplot
    cols = 3
    rows = (len(sample_images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows))
    
    if rows == 1:
        axes = axes.reshape(1, -1)
    
    fig.suptitle(f"Sample Images from Class: {class_name} ({split})", fontsize=16, fontweight='bold')
    
    for idx, img_file in enumerate(sample_images):
        row = idx // cols
        col = idx % cols
        
        try:
            img_path = os.path.join(class_path, img_file)
            img = Image.open(img_path)
            
            axes[row, col].imshow(img)
            axes[row, col].set_title(f"{img_file}\nSize: {img.size}", fontsize=10)
            axes[row, col].axis('off')
            
        except Exception as e:
            axes[row, col].text(0.5, 0.5, f"Error loading\n{img_file}", 
                               ha='center', va='center', transform=axes[row, col].transAxes)
            axes[row, col].axis('off')
    
    # Hide empty subplots
    for idx in range(len(sample_images), rows * cols):
        row = idx // cols
        col = idx % cols
        axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"📊 Displayed {len(sample_images)} out of {len(image_files)} images from class '{class_name}'")

# Example: Display images from the first available class
if class_to_idx:
    first_class = list(class_to_idx.keys())[0]
    print(f"🖼️ Displaying sample images from class: {first_class}")
    display_sample_images(first_class, num_images=6)
else:
    print("⚠️ No classes available. Please check your dataset path.")

In [ ]:
# Image Metadata Analysis
def analyze_image_metadata(class_name, max_images=50, split='train'):
    """Analyze metadata for images in a specific class"""
    
    data_dir = TRAIN_DIR if split == 'train' else VAL_DIR
    class_path = os.path.join(data_dir, class_name)
    
    if not os.path.exists(class_path):
        print(f"❌ Class '{class_name}' not found")
        return None
    
    image_files = [f for f in os.listdir(class_path) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    
    sample_files = random.sample(image_files, min(max_images, len(image_files)))
    
    metadata = {
        'filenames': [],
        'widths': [],
        'heights': [],
        'file_sizes': [],
        'formats': [],
        'modes': [],
        'aspect_ratios': []
    }
    
    print(f"🔍 Analyzing metadata for {len(sample_files)} images from class '{class_name}'...")
    
    for img_file in sample_files:
        try:
            img_path = os.path.join(class_path, img_file)
            
            # File size
            file_size = os.path.getsize(img_path)
            
            # Image properties
            with Image.open(img_path) as img:
                width, height = img.size
                format_type = img.format
                mode = img.mode
                aspect_ratio = width / height
                
                metadata['filenames'].append(img_file)
                metadata['widths'].append(width)
                metadata['heights'].append(height)
                metadata['file_sizes'].append(file_size)
                metadata['formats'].append(format_type)
                metadata['modes'].append(mode)
                metadata['aspect_ratios'].append(aspect_ratio)
                
        except Exception as e:
            print(f"⚠️ Error processing {img_file}: {e}")
    
    return metadata

def visualize_metadata(metadata, class_name):
    """Visualize image metadata"""
    
    if not metadata or len(metadata['widths']) == 0:
        print("❌ No metadata to visualize")
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f"Image Metadata Analysis - Class: {class_name}", fontsize=16, fontweight='bold')
    
    # Width distribution
    axes[0, 0].hist(metadata['widths'], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0, 0].set_title('Width Distribution')
    axes[0, 0].set_xlabel('Width (pixels)')
    axes[0, 0].set_ylabel('Frequency')
    
    # Height distribution
    axes[0, 1].hist(metadata['heights'], bins=20, alpha=0.7, color='lightgreen', edgecolor='black')
    axes[0, 1].set_title('Height Distribution')
    axes[0, 1].set_xlabel('Height (pixels)')
    axes[0, 1].set_ylabel('Frequency')
    
    # File size distribution
    file_sizes_mb = [size / (1024*1024) for size in metadata['file_sizes']]
    axes[0, 2].hist(file_sizes_mb, bins=20, alpha=0.7, color='coral', edgecolor='black')
    axes[0, 2].set_title('File Size Distribution')
    axes[0, 2].set_xlabel('File Size (MB)')
    axes[0, 2].set_ylabel('Frequency')
    
    # Aspect ratio distribution
    axes[1, 0].hist(metadata['aspect_ratios'], bins=20, alpha=0.7, color='gold', edgecolor='black')
    axes[1, 0].set_title('Aspect Ratio Distribution')
    axes[1, 0].set_xlabel('Aspect Ratio (W/H)')
    axes[1, 0].set_ylabel('Frequency')
    
    # Format distribution
    format_counts = Counter(metadata['formats'])
    axes[1, 1].bar(format_counts.keys(), format_counts.values(), color='lightpink', edgecolor='black')
    axes[1, 1].set_title('Image Format Distribution')
    axes[1, 1].set_xlabel('Format')
    axes[1, 1].set_ylabel('Count')
    
    # Mode distribution
    mode_counts = Counter(metadata['modes'])
    axes[1, 2].bar(mode_counts.keys(), mode_counts.values(), color='lightcyan', edgecolor='black')
    axes[1, 2].set_title('Color Mode Distribution')
    axes[1, 2].set_xlabel('Mode')
    axes[1, 2].set_ylabel('Count')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print(f"\n📊 Metadata Summary for class '{class_name}':")
    print(f"📏 Width: min={min(metadata['widths'])}, max={max(metadata['widths'])}, avg={np.mean(metadata['widths']):.1f}")
    print(f"📏 Height: min={min(metadata['heights'])}, max={max(metadata['heights'])}, avg={np.mean(metadata['heights']):.1f}")
    print(f"💾 File size: min={min(file_sizes_mb):.2f}MB, max={max(file_sizes_mb):.2f}MB, avg={np.mean(file_sizes_mb):.2f}MB")
    print(f"📐 Aspect ratio: min={min(metadata['aspect_ratios']):.2f}, max={max(metadata['aspect_ratios']):.2f}, avg={np.mean(metadata['aspect_ratios']):.2f}")

# Example: Analyze metadata for the first available class
if class_to_idx:
    first_class = list(class_to_idx.keys())[0]
    metadata = analyze_image_metadata(first_class, max_images=30)
    if metadata:
        visualize_metadata(metadata, first_class)
else:
    print("⚠️ No classes available for metadata analysis.")

In [ ]:
# Interactive Image Viewer
if WIDGETS_AVAILABLE:
    
    class ImageNetExplorer:
        def __init__(self):
            self.current_images = []
            self.current_class = None
            self.current_index = 0
            
        def create_explorer_widget(self):
            """Create interactive explorer widget"""
            
            # Class dropdown
            available_classes = list(class_to_idx.keys())[:20]  # Limit for performance
            self.class_dropdown = widgets.Dropdown(
                options=available_classes,
                value=available_classes[0] if available_classes else None,
                description='Class:',
                style={'description_width': 'initial'}
            )
            
            # Split dropdown
            self.split_dropdown = widgets.Dropdown(
                options=['train', 'val'],
                value='train',
                description='Split:',
                style={'description_width': 'initial'}
            )
            
            # Navigation buttons
            self.prev_button = widgets.Button(description='◀ Previous', button_style='info')
            self.next_button = widgets.Button(description='Next ▶', button_style='info')
            self.random_button = widgets.Button(description='🎲 Random', button_style='warning')
            
            # Image info
            self.image_info = widgets.HTML(value="Select a class to start exploring")
            
            # Image display
            self.image_output = widgets.Output()
            
            # Set up event handlers
            self.class_dropdown.observe(self.on_class_change, names='value')
            self.split_dropdown.observe(self.on_split_change, names='value')
            self.prev_button.on_click(self.show_previous)
            self.next_button.on_click(self.show_next)
            self.random_button.on_click(self.show_random)
            
            # Layout
            controls = widgets.HBox([self.class_dropdown, self.split_dropdown])
            navigation = widgets.HBox([self.prev_button, self.next_button, self.random_button])
            
            return widgets.VBox([controls, navigation, self.image_info, self.image_output])
        
        def load_class_images(self, class_name, split):
            """Load images for a specific class"""
            data_dir = TRAIN_DIR if split == 'train' else VAL_DIR
            class_path = os.path.join(data_dir, class_name)
            
            if os.path.exists(class_path):
                self.current_images = [f for f in os.listdir(class_path) 
                                     if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
                self.current_class = class_name
                self.current_index = 0
                return True
            return False
        
        def on_class_change(self, change):
            """Handle class selection change"""
            if self.load_class_images(change['new'], self.split_dropdown.value):
                self.show_current_image()
        
        def on_split_change(self, change):
            """Handle split selection change"""
            if self.current_class and self.load_class_images(self.current_class, change['new']):
                self.show_current_image()
        
        def show_previous(self, button):
            """Show previous image"""
            if self.current_images:
                self.current_index = (self.current_index - 1) % len(self.current_images)
                self.show_current_image()
        
        def show_next(self, button):
            """Show next image"""
            if self.current_images:
                self.current_index = (self.current_index + 1) % len(self.current_images)
                self.show_current_image()
        
        def show_random(self, button):
            """Show random image"""
            if self.current_images:
                self.current_index = random.randint(0, len(self.current_images) - 1)
                self.show_current_image()
        
        def show_current_image(self):
            """Display current image with info"""
            with self.image_output:
                self.image_output.clear_output(wait=True)
                
                if not self.current_images:
                    print("No images found for this class")
                    return
                
                try:
                    data_dir = TRAIN_DIR if self.split_dropdown.value == 'train' else VAL_DIR
                    img_path = os.path.join(data_dir, self.current_class, self.current_images[self.current_index])
                    
                    img = Image.open(img_path)
                    
                    # Display image
                    plt.figure(figsize=(10, 8))
                    plt.imshow(img)
                    plt.axis('off')
                    plt.title(f"{self.current_images[self.current_index]}", fontsize=14, fontweight='bold')
                    plt.tight_layout()
                    plt.show()
                    
                    # Update info
                    file_size = os.path.getsize(img_path) / (1024*1024)
                    info_html = f'''
                    <div style="font-family: Arial, sans-serif; background-color: #f0f0f0; padding: 10px; border-radius: 5px;">
                        <h4>Image Information</h4>
                        <p><strong>Class:</strong> {self.current_class}</p>
                        <p><strong>Filename:</strong> {self.current_images[self.current_index]}</p>
                        <p><strong>Image {self.current_index + 1} of {len(self.current_images)}</strong></p>
                        <p><strong>Dimensions:</strong> {img.size[0]} × {img.size[1]} pixels</p>
                        <p><strong>Format:</strong> {img.format}</p>
                        <p><strong>Mode:</strong> {img.mode}</p>
                        <p><strong>File Size:</strong> {file_size:.2f} MB</p>
                    </div>
                    '''
                    self.image_info.value = info_html
                    
                except Exception as e:
                    print(f"Error loading image: {e}")
    
    # Create and display the explorer
    explorer = ImageNetExplorer()
    explorer_widget = explorer.create_explorer_widget()
    
    print("🎛️ Interactive ImageNet Explorer")
    print("Use the dropdown to select a class and navigate through images!")
    display(explorer_widget)
    
else:
    print("⚠️ Interactive widgets not available. Install ipywidgets to enable the interactive explorer.")
    print("Run: pip install ipywidgets")

In [ ]:
# Class Distribution Visualization
def visualize_class_distribution(max_classes=50):
    """Visualize the distribution of images across classes"""
    
    print(f"📊 Analyzing class distribution (max {max_classes} classes)...")
    
    train_counts = {}
    val_counts = {}
    
    # Count images in training set
    if os.path.exists(TRAIN_DIR):
        train_classes = sorted(os.listdir(TRAIN_DIR))[:max_classes]
        for class_dir in train_classes:
            class_path = os.path.join(TRAIN_DIR, class_dir)
            if os.path.isdir(class_path):
                image_count = len([f for f in os.listdir(class_path) 
                                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
                train_counts[class_dir] = image_count
    
    # Count images in validation set
    if os.path.exists(VAL_DIR):
        val_classes = sorted(os.listdir(VAL_DIR))[:max_classes]
        for class_dir in val_classes:
            class_path = os.path.join(VAL_DIR, class_dir)
            if os.path.isdir(class_path):
                image_count = len([f for f in os.listdir(class_path) 
                                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
                val_counts[class_dir] = image_count
    
    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    fig.suptitle('ImageNet Class Distribution Analysis', fontsize=16, fontweight='bold')
    
    # Training set bar chart
    if train_counts:
        classes = list(train_counts.keys())
        counts = list(train_counts.values())
        
        axes[0, 0].bar(range(len(classes)), counts, color='skyblue', alpha=0.7)
        axes[0, 0].set_title(f'Training Set - Images per Class (First {len(classes)} classes)')
        axes[0, 0].set_xlabel('Class Index')
        axes[0, 0].set_ylabel('Number of Images')
        axes[0, 0].tick_params(axis='x', rotation=45)
        
        # Training set histogram
        axes[0, 1].hist(counts, bins=20, color='skyblue', alpha=0.7, edgecolor='black')
        axes[0, 1].set_title('Training Set - Distribution of Image Counts')
        axes[0, 1].set_xlabel('Number of Images per Class')
        axes[0, 1].set_ylabel('Number of Classes')
    
    # Validation set bar chart
    if val_counts:
        classes = list(val_counts.keys())
        counts = list(val_counts.values())
        
        axes[1, 0].bar(range(len(classes)), counts, color='lightgreen', alpha=0.7)
        axes[1, 0].set_title(f'Validation Set - Images per Class (First {len(classes)} classes)')
        axes[1, 0].set_xlabel('Class Index')
        axes[1, 0].set_ylabel('Number of Images')
        axes[1, 0].tick_params(axis='x', rotation=45)
        
        # Validation set histogram
        axes[1, 1].hist(counts, bins=20, color='lightgreen', alpha=0.7, edgecolor='black')
        axes[1, 1].set_title('Validation Set - Distribution of Image Counts')
        axes[1, 1].set_xlabel('Number of Images per Class')
        axes[1, 1].set_ylabel('Number of Classes')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    if train_counts:
        train_values = list(train_counts.values())
        print(f"\n📈 Training Set Statistics ({len(train_counts)} classes):")
        print(f"   Total images: {sum(train_values):,}")
        print(f"   Min images per class: {min(train_values)}")
        print(f"   Max images per class: {max(train_values)}")
        print(f"   Average images per class: {np.mean(train_values):.1f}")
        print(f"   Std deviation: {np.std(train_values):.1f}")
    
    if val_counts:
        val_values = list(val_counts.values())
        print(f"\n📈 Validation Set Statistics ({len(val_counts)} classes):")
        print(f"   Total images: {sum(val_values):,}")
        print(f"   Min images per class: {min(val_values)}")
        print(f"   Max images per class: {max(val_values)}")
        print(f"   Average images per class: {np.mean(val_values):.1f}")
        print(f"   Std deviation: {np.std(val_values):.1f}")

# Run the visualization
visualize_class_distribution(max_classes=30)

In [ ]:
# Image Statistics and Properties
def analyze_image_properties(class_name, max_images=100, split='train'):
    """Analyze color and pixel properties of images"""
    
    data_dir = TRAIN_DIR if split == 'train' else VAL_DIR
    class_path = os.path.join(data_dir, class_name)
    
    if not os.path.exists(class_path):
        print(f"❌ Class '{class_name}' not found")
        return
    
    image_files = [f for f in os.listdir(class_path) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    
    sample_files = random.sample(image_files, min(max_images, len(image_files)))
    
    print(f"🎨 Analyzing color properties for {len(sample_files)} images from class '{class_name}'...")
    
    # Collect color statistics
    red_means = []
    green_means = []
    blue_means = []
    brightness_values = []
    contrast_values = []
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f'Image Properties Analysis - Class: {class_name}', fontsize=16, fontweight='bold')
    
    # Sample a few images for detailed analysis
    sample_for_hist = random.sample(sample_files, min(5, len(sample_files)))
    combined_red = []
    combined_green = []
    combined_blue = []
    
    for img_file in sample_files:
        try:
            img_path = os.path.join(class_path, img_file)
            img = Image.open(img_path)
            
            # Convert to RGB if necessary
            if img.mode != 'RGB':
                img = img.convert('RGB')
            
            # Convert to numpy array
            img_array = np.array(img)
            
            # Calculate channel means
            red_mean = np.mean(img_array[:, :, 0])
            green_mean = np.mean(img_array[:, :, 1])
            blue_mean = np.mean(img_array[:, :, 2])
            
            red_means.append(red_mean)
            green_means.append(green_mean)
            blue_means.append(blue_mean)
            
            # Calculate brightness (luminance)
            brightness = 0.299 * red_mean + 0.587 * green_mean + 0.114 * blue_mean
            brightness_values.append(brightness)
            
            # Calculate contrast (standard deviation of grayscale)
            gray = 0.299 * img_array[:, :, 0] + 0.587 * img_array[:, :, 1] + 0.114 * img_array[:, :, 2]
            contrast = np.std(gray)
            contrast_values.append(contrast)
            
            # Collect pixel values for histogram (sample images)
            if img_file in sample_for_hist:
                combined_red.extend(img_array[:, :, 0].flatten())
                combined_green.extend(img_array[:, :, 1].flatten())
                combined_blue.extend(img_array[:, :, 2].flatten())
        
        except Exception as e:
            print(f"⚠️ Error processing {img_file}: {e}")
    
    # Plot RGB channel means
    axes[0, 0].hist([red_means, green_means, blue_means], bins=20, alpha=0.7, 
                   label=['Red', 'Green', 'Blue'], color=['red', 'green', 'blue'])
    axes[0, 0].set_title('Average RGB Channel Values')
    axes[0, 0].set_xlabel('Average Pixel Value')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].legend()
    
    # Plot brightness distribution
    axes[0, 1].hist(brightness_values, bins=20, alpha=0.7, color='gold', edgecolor='black')
    axes[0, 1].set_title('Brightness Distribution')
    axes[0, 1].set_xlabel('Brightness')
    axes[0, 1].set_ylabel('Frequency')
    
    # Plot contrast distribution
    axes[0, 2].hist(contrast_values, bins=20, alpha=0.7, color='purple', edgecolor='black')
    axes[0, 2].set_title('Contrast Distribution')
    axes[0, 2].set_xlabel('Contrast (Std Dev)')
    axes[0, 2].set_ylabel('Frequency')
    
    # Plot combined pixel histograms
    if combined_red:
        axes[1, 0].hist(combined_red, bins=50, alpha=0.7, color='red', edgecolor='none')
        axes[1, 0].set_title('Combined Red Channel Histogram')
        axes[1, 0].set_xlabel('Pixel Value')
        axes[1, 0].set_ylabel('Frequency')
        
        axes[1, 1].hist(combined_green, bins=50, alpha=0.7, color='green', edgecolor='none')
        axes[1, 1].set_title('Combined Green Channel Histogram')
        axes[1, 1].set_xlabel('Pixel Value')
        axes[1, 1].set_ylabel('Frequency')
        
        axes[1, 2].hist(combined_blue, bins=50, alpha=0.7, color='blue', edgecolor='none')
        axes[1, 2].set_title('Combined Blue Channel Histogram')
        axes[1, 2].set_xlabel('Pixel Value')
        axes[1, 2].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print(f"\n🎨 Color Properties Summary for class '{class_name}':")
    print(f"🔴 Red channel: mean={np.mean(red_means):.1f}, std={np.std(red_means):.1f}")
    print(f"🟢 Green channel: mean={np.mean(green_means):.1f}, std={np.std(green_means):.1f}")
    print(f"🔵 Blue channel: mean={np.mean(blue_means):.1f}, std={np.std(blue_means):.1f}")
    print(f"💡 Brightness: mean={np.mean(brightness_values):.1f}, std={np.std(brightness_values):.1f}")
    print(f"🌈 Contrast: mean={np.mean(contrast_values):.1f}, std={np.std(contrast_values):.1f}")

# Example: Analyze properties for the first available class
if class_to_idx:
    first_class = list(class_to_idx.keys())[0]
    analyze_image_properties(first_class, max_images=50)
else:
    print("⚠️ No classes available for property analysis.")

## 🛠️ Utility Functions

The notebook provides several utility functions you can use for further exploration:

### Available Functions:
- `display_sample_images(class_name, num_images, split)` - Display sample images from a class
- `analyze_image_metadata(class_name, max_images, split)` - Analyze image file properties
- `analyze_image_properties(class_name, max_images, split)` - Analyze color and pixel properties
- `visualize_class_distribution(max_classes)` - Show distribution across classes

### Interactive Explorer:
If ipywidgets is available, use the interactive explorer above to browse images by class with:
- Class selection dropdown
- Train/validation split selection  
- Navigation buttons (Previous, Next, Random)
- Detailed image information display

### Tips for Usage:
1. **Update the dataset path** in the second cell to point to your ImageNet location
2. **Start with small samples** when analyzing many images to avoid long processing times
3. **Use the interactive explorer** for quick browsing and inspection
4. **Analyze metadata first** to understand file formats and sizes before diving into pixel analysis
5. **Compare classes** by running the same analysis on different class names

### Next Steps:
- Identify classes with unusual properties (size, color distribution, etc.)
- Compare training vs validation set characteristics
- Use insights for data preprocessing decisions
- Select representative classes for model testing